Thunderbird 원본(약 2.1억 줄)에서 v7 규격대로 Alert-Normal 짝을 뽑는 스트리밍 파서.
* "alert가 normal보다 어텐션을 더 받나?" (S_Alert > S_Normal)에 답을 하기 위한 코드

핵심 원칙
- alert(첫 칸이 '-'가 아닌 줄)만 수집하되, 연속 burst는 하나의 incident로 묶는다 (v7 §2).
- 각 incident alert target에 같은 node의 Normal target 1개를 매칭한다
  (v7 §4: same node 필수 + 시간/토큰길이 통제).

출력
- pairs.csv : 짝 목록 + 매칭 품질

이 스크립트는 "짝 추출"까지만 한다.
21-line 창 만들기(v7문서에 명시), 파일럿 임계값(2번), 시뮬 검정력(1번)은
여기서 나온 pairs.csv로 다음 단계에서 이어서 한다.

In [13]:
import csv, re
from collections import deque, Counter
from pathlib import Path
from transformers import AutoTokenizer

Raw_path = Path("/Users/mac/Downloads/Thunderbird.log")
Out_path = Path("pairs_out/pairs.csv")
MODEL = "Qwen/Qwen2.5-3B-Instruct"

Burts_sec = 3600
match_time_sec = 3600
per_tempalte = 100
node_buf = 400
seed = 42

Out_path.parent.mkdir(parents=True, exist_ok = True)
tok = AutoTokenizer.from_pretrained(MODEL)

In [14]:
# 템플릿 만들기
#p[0]: 라벨(-) | p[1] or ts : 타임스템프 | p[2] : 날짜 | p[3] : 노드


def n_tok(text): # 토큰 개수 세기. autotikenizer사용. input_ids는 내부적으로 저장된 형태(토크나이저 고유 번호)
    return len(tok(text, add_special_tokens=False)["input_ids"])
# add_sepecial_tokens 순수한 토큰개수만 세기.

_num = re.compile(r"0*[0-9a-fA-F]+|-?\d+") #템플릿 모양

def parse(raw):
    p = raw.split()
    if len(p) < 5: #최소 5조각은 있어야 함.  
        return None
    try:
        ts = int(p[1])
    except ValueError:
        return None # 8조각으로 나눔. 8개 이상의 조각은 정상적인 로그이므로, 마지막 덩어리(진짜 메시지)만 빼서 저장.
    content = raw.split(None, 8)[-1].rstrip("\n") if len(p) > 8 else raw.rstrip("\n")
    return p[0], ts, p[3], content

def template(content): # 템플릿 추출
    return _num.sub("<*>", content)    

In [15]:
# alert = 맨 앞칸에 이상 라벨이 붙은 줄 (뭔가 문제 있는 줄)
# normal = 맨 앞칸이 **-**인 줄 (평범한 정상 로그)
# alert가 이미 주어졌다고 치고(a_idx, a_ts...) 맞는 normal을 찾기

node_recent = {} # 짝 매칭을 위한 버퍼용.
taken = Counter()
active = None
rows = []


def match_normal(a_idx, a_ts, a_node, a_content):
    a_tokens = n_tok(a_content)
    cands = [r for r in node_recent.get(a_node, []) # value 꺼내기    # abs: 순수 시간 거리 측정.
        if r[2] == "-" and r[0] != a_idx and abs(r[1] - a_ts) <= match_time_sec] # 매칭 유효기간(timeout 설정)
    if not cands: # node_recent가 비어있거나 조건에 맞는 짝이 없을 경우
        return None, a_tokens, None
    cands.sort(key=lambda r: (abs(len(r[3]) - len(a_content)), abs(r[1] - a_ts)))
    # 텍스트 길이 비교 후 절대값이 0에 가까울수록 맨 앞줄로 보냄. / 동점이라면 시간차이가 더 짧은것으로.
    best, best_key = None, None
    for r in cands[:20]: # 상위 20등만 검사(예비 20짝만 추출)
        key = (abs(n_tok(r[3]) - a_tokens), abs(r[1] - a_ts)) # 이제 실제 토큰과 검사
        if best_key is None or key < best_key:
            best_key, best = key, r #최적의 짝 갱신
    return best, a_tokens, n_tok(best[3])

In [ ]:
with Raw_path.open(encoding="utf-8", errors="replace") as f:
    for idx, raw in enumerate(f):
        rec = parse(raw)
        if rec is None:
            continue
        label, ts, node, content = rec  # 앞선 p리스트 값 나누기
        
        
        buf = node_recent.setdefault(node, deque(maxlen=node_buf)) # 정해진 용량 초과 방지
        buf.append((idx,ts,label, content)) # 노드별 보관함. 나중에 짝으로 쓸 후보 창고
        
        if label == "-": # normal이면 continue
            continue
        
        sig = template(content) # 템플릿 변환
        
        
#         줄 하나 읽기
#  → 이상하면 버림
#  → 보관함에 넣기 (노드별)
#  → normal이면 끝 (다음 줄)
#  → alert면:
#      → 방금과 같은 사건이면 넘김 (burst)
#      → 종류 100개 넘었으면 넘김
#      → 아니면 normal 짝 찾아서 저장

        if active and active[0] == node and active[1] == sig and ts-active[2] <= Burts_sec:
            active = (node, sig, ts)
            continue # 1시간에 수백개 에러가 터지면, 무시(같은 원인 에러, 로그의 특성)
        active = (node, sig, ts)
        
        if taken[sig] >= per_tempalte: # 종류가 100개 넘었으면 넘김. 왜? 종류개수를 한정해두는 이유는? -> 자주 터지는 에러의 독식 방지. 상한이 없으면 한 종류만 차지하게 됨.
            continue
        taken[sig] += 1
        
        normal, a_tokens, n_tokens = match_normal(idx, ts, node, content) # alert - normal 매칭
        pid = f"P{len(rows):05d}"
        if normal is None: # nomal이 없으면 비워두기
            rows.append([pid, "no_normal", sig[:80], idx, ts, node, a_tokens, 
                         "", "", "", "", content[:200]])
        else: # 아니면 형식에 맞게 저장.
            rows.append([pid, "ok", sig[:80], idx, ts, node,
                         a_tokens, normal[0], normal[1], n_tokens,
                         abs(normal[1] - ts), content[:200]])

In [17]:
header = ["pair_id", "status", "template", "alert_idx", "alert_ts", "node",
          "alert_tokens", "normal_idx", "normal_ts", "normal_tokens",
          "dt_sec", "alert_content"]
with Out_path.open("w", newline="", encoding="utf-8-sig") as fp:
    w = csv.writer(fp)
    w.writerow(header)
    w.writerows(rows)
 
ok = sum(1 for r in rows if r[1] == "ok")
print(f"짝 성공: {ok}   |   normal 없음: {len(rows) - ok}   |   종류: {len(taken)}")
print(f"저장: {Out_path}")

짝 성공: 1056   |   normal 없음: 46   |   종류: 43
저장: pairs_out/pairs.csv
